In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import asyncio
import websockets
import json
import threading
import queue

SERVER_URL = "ws://localhost:8000/api/ws/video"

# ================= Mediapipe setup =================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

# Queue dùng để trao đổi dữ liệu giữa thread
feature_queue = queue.Queue(maxsize=10)

def extract_features(frame):
    """Trích xuất vector 42D từ frame"""
    h, w, _ = frame.shape
    results = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if results.multi_hand_landmarks:
        lm = results.multi_hand_landmarks[0]
        coords = []
        for p in lm.landmark:
            coords.extend([p.x * w, p.y * h])
        return np.array(coords, dtype=np.float32)
    else:
        return np.zeros(42, dtype=np.float32)  # padding nếu ko có tay

# ================= Thread A: Capture & Feature Extraction =================
def camera_thread():
    cap = cv2.VideoCapture(0)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (320, 240))
        feat = extract_features(frame)

        print("Feature:", feat.tolist())

        # push feature vào queue (nếu đầy thì bỏ bớt)
        try:
            feature_queue.put_nowait(feat.tolist())
        except queue.Full:
            pass

        cv2.imshow("Client", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

# ================= Thread B: Async WebSocket =================
async def send_features():
    async with websockets.connect(SERVER_URL) as ws:
        while True:
            # lấy feature từ queue (blocking nhưng không chiếm CPU)
            feat = await asyncio.get_event_loop().run_in_executor(None, feature_queue.get)
            
            # ép sang numpy rồi gửi dạng binary
            feat = np.array(feat, dtype=np.float32)
            await ws.send(feat.tobytes())

            # nhận response nếu có
            try:
                response = await asyncio.wait_for(ws.recv(), timeout=0.1)
                print("Server:", response)
            except asyncio.TimeoutError:
                pass

# ================= Main =================
if __name__ == "__main__":
    # chạy camera thread song song
    t = threading.Thread(target=camera_thread, daemon=True)
    t.start()

    # chạy websocket loop
    await send_features()


d:\Github\Machine-Learning-Studies\tf-env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Feature: [252.7149658203125, 211.24195861816406, 231.18585205078125, 198.95248413085938, 219.46630859375, 178.54058837890625, 209.432861328125, 163.5081024169922, 199.91143798828125, 147.64276123046875, 241.50277709960938, 150.89273071289062, 226.91610717773438, 137.59933471679688, 209.37020874023438, 137.02658081054688, 195.68878173828125, 141.20924377441406, 249.3486328125, 148.39450073242188, 226.48231506347656, 131.8094024658203, 206.89089965820312, 137.9647216796875, 195.6460418701172, 146.81488037109375, 257.3053894042969, 149.10328674316406, 226.003662109375, 142.15077209472656, 216.61917114257812, 163.2579803466797, 217.8853759765625, 180.90399169921875, 263.3744812011719, 153.83453369140625, 235.71292114257812, 148.25897216796875, 225.15188598632812, 161.9775390625, 223.73191833496094, 176.15823364257812]
Feature: [271.0423583984375, 205.06983947753906, 246.26898193359375, 192.99806213378906, 227.40963745117188, 171.91964721679688, 214.60020446777344, 153.9791717529297, 206.58

AttributeError: 'list' object has no attribute 'astype'

Feature: [303.07244873046875, 205.16104125976562, 273.11572265625, 192.15420532226562, 253.32574462890625, 160.8887481689453, 247.0561981201172, 133.22940063476562, 242.1723175048828, 112.43826293945312, 271.7591857910156, 126.83418273925781, 269.73345947265625, 99.65546417236328, 270.7370300292969, 85.17431640625, 272.2727355957031, 73.27177429199219, 291.8770751953125, 122.96931457519531, 294.5672607421875, 89.93035888671875, 297.90399169921875, 70.04407501220703, 301.7914123535156, 54.92171859741211, 309.0610656738281, 126.86851501464844, 317.3489685058594, 99.57510375976562, 319.89398193359375, 84.5982894897461, 321.94482421875, 72.29710388183594, 325.40582275390625, 136.092529296875, 331.8564453125, 115.49275207519531, 334.58380126953125, 102.67173767089844, 336.65087890625, 90.79987335205078]
Feature: [307.36639404296875, 207.1376953125, 276.0516662597656, 192.76268005371094, 257.39251708984375, 159.94126892089844, 253.04197692871094, 130.65744018554688, 254.98989868164062, 111.5

In [2]:
run_all()

d:\Github\Machine-Learning-Studies\tf-env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


SystemExit: 0

d:\Github\Machine-Learning-Studies\tf-env\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
